In [ ]:
%cd /content/FastSpeech2
!git pull

[Errno 2] No such file or directory: '/content/FastSpeech2'
/content
fatal: not a git repository (or any of the parent directories): .git


In [1]:
import os, glob, zipfile
from google.colab import drive
drive.mount('/content/drive')

DRIVE = "/content/drive/MyDrive/fs2_bg_phone"
STEP  = 60000

# repos
os.chdir("/content")
if not os.path.isdir("FastSpeech2"):
    os.system("git clone -b vocoder-finetune https://github.com/StankoI/FastSpeech2.git")
if not os.path.isdir("hifi-gan"):
    os.system("git clone https://github.com/jik876/hifi-gan")
os.chdir("/content/FastSpeech2")
os.system("unzip -o -q hifigan/generator_universal.pth.tar.zip -d hifigan/")

# data: features + wavs + the GTA mels you already generated (all from Drive)
if not glob.glob("/content/data/preprocessed/**/stats.json", recursive=True):
    with zipfile.ZipFile(f"{DRIVE}/preprocessed_Bulgarian_prosody_v2.zip") as z:
        z.extractall("/content/data/preprocessed")
if not glob.glob("/content/data/raw/**/*.wav", recursive=True):
    with zipfile.ZipFile(f"{DRIVE}/raw_data_Bulgarian.zip") as z:
        z.extractall("/content/data/raw")
if not glob.glob("/content/FastSpeech2/gta_mels/Bulgarian/*.npy"):
    os.system(f"tar -C /content/FastSpeech2 -xf '{DRIVE}/gta_mels_Bulgarian.tar'")

PP   = os.path.dirname(glob.glob("/content/data/preprocessed/**/stats.json", recursive=True)[0])
WAVS = os.path.dirname(glob.glob("/content/data/raw/**/*.wav", recursive=True)[0])
GTA  = "/content/FastSpeech2/gta_mels/Bulgarian"

os.makedirs("preprocessed_data", exist_ok=True); os.makedirs("output/ckpt", exist_ok=True)
os.system(f"ln -sfn '{PP}' preprocessed_data/Bulgarian")
os.system(f"ln -sfn '{DRIVE}/output_prosody_v2/ckpt/Bulgarian' output/ckpt/Bulgarian")

print("WAVS =", WAVS)
print("GTA  =", GTA, "->", len(glob.glob(f"{GTA}/*.npy")), "mels")
print("features:", os.path.exists("preprocessed_data/Bulgarian/stats.json"))

Mounted at /content/drive
WAVS = /content/data/raw/Bulgarian/Bulgarian
GTA  = /content/FastSpeech2/gta_mels/Bulgarian -> 44383 mels
features: True


Align wav with generated mel

In [15]:
import os
os.chdir("/content/FastSpeech2")
TRIM_WAVS = "/content/FastSpeech2/gta_wavs/Bulgarian"
os.system(
    "python tools/dump_gta_wavs.py -p config/Bulgarian/preprocess.yaml "
    f"--gta_dir '{GTA}' --raw_wav_dir '{WAVS}' --out_dir '{TRIM_WAVS}' --filelists train.txt,val.txt"
)

0

Build HiFi-GAN filelists

In [ ]:
import os, glob, numpy as np
os.chdir("/content/FastSpeech2")
TRIM_IDS = {os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f"{TRIM_WAVS}/*.wav")}
FPS = 8192 // 256  # frames per training segment = 32

def build(meta, dst, min_frames=0, max_frames=None, limit=None):
    ids  = [ln.split("|",1)[0] for ln in open(f"preprocessed_data/Bulgarian/{meta}", encoding="utf-8") if ln.strip()]
    keep = []
    for i in ids:
        if i not in TRIM_IDS: continue
        T = np.load(f"{GTA}/{i}.npy").shape[1]
        if min_frames and T < min_frames: continue
        if max_frames and T > max_frames: continue
        keep.append(i)
        if limit and len(keep) >= limit: break
    open(dst, "w", encoding="utf-8").write("\n".join(keep))
    print(f"{meta}: kept {len(keep)}/{len(ids)}")

build("train.txt", "/content/hifi-gan/training.txt",   min_frames=FPS+2)            # >=34 frames
build("val.txt",   "/content/hifi-gan/validation.txt", max_frames=1200, limit=200)  # short subset

train.txt: kept 43670/43883
val.txt: kept 200/500


Checkpoint dirs

In [2]:
import os, glob
CP = f"{DRIVE}/hifigan_finetune"      # checkpoints on Drive -> resumable across sessions
os.makedirs(CP, exist_ok=True)

if not glob.glob(f"{CP}/g_*"):        # idempotent: only seed on a fresh run
    os.system(
        "python /content/FastSpeech2/tools/seed_hifigan_finetune.py "
        "--hifigan_repo /content/hifi-gan "
        "--universal /content/FastSpeech2/hifigan/generator_universal.pth.tar "
        "--config /content/FastSpeech2/hifigan/config.json "
        f"--out_dir '{CP}'"
    )
else:
    print("checkpoints already in", CP, "- skipping seed, will resume from latest")
print(sorted(os.listdir(CP))[:6])

checkpoints already in /content/drive/MyDrive/fs2_bg_phone/hifigan_finetune - skipping seed, will resume from latest
['config.json', 'do_00000000', 'do_00005000', 'do_00010000', 'do_00015000', 'do_00020000']


### Train


In [ ]:
p = "/content/hifi-gan/meldataset.py"
s = open(p).read(); orig = s
s = s.replace(
    "librosa_mel_fn(sampling_rate, n_fft, num_mels, fmin, fmax)",
    "librosa_mel_fn(sr=sampling_rate, n_fft=n_fft, n_mels=num_mels, fmin=fmin, fmax=fmax)",
)
if "return_complex" not in s:
    s = s.replace("onesided=True)", "onesided=True, return_complex=False)")
open(p, "w").write(s)
print("meldataset patched:", orig != s)

meldataset patched: False


In [ ]:
%cd /content/hifi-gan
!python -u train.py --config /content/FastSpeech2/hifigan/config.json --input_wavs_dir "{TRIM_WAVS}" --input_mels_dir "{GTA}" --input_training_file training.txt --input_validation_file validation.txt --checkpoint_path "{CP}" --fine_tuning True --stdout_interval 100 --checkpoint_interval 5000 --summary_interval 100 --validation_interval 5000 --training_epochs 200

/content/hifi-gan
2026-06-23 17:34:04.316535: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-23 17:34:04.387482: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Initializing Training Process..
Batch size per GPU : 16
Generator(
  (conv_pre): Conv1d(80, 512, kernel_size=(7,), stride=(1,), padding=(3,))
  (ups): ModuleList(
    (0): ConvTranspose1d(512, 256, kernel_size=(16,), stride=(8,), padding=(4,))
    (1): ConvTranspose1d(256, 128, kernel_size=(16,), stride=(8,), padding=(4,))
    (2): ConvTranspose1d(128, 6

In [ ]:
import shutil, glob, os
os.makedirs(f"/content/drive/MyDrive/g_backup", exist_ok=True)
for s in [50000, 85000]:                      # + any checkpoints you liked
    shutil.copy(f"{CP}/g_{s:08d}", f"/content/drive/MyDrive/g_backup")

In [ ]:
import os, sys, glob, io, json, zipfile, yaml, torch, numpy as np
from IPython.display import Audio, display
from scipy.io import wavfile

os.chdir("/content/FastSpeech2")
sys.path.insert(0, "/content/FastSpeech2")
import hifigan   # now resolvable

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pc = yaml.load(open("config/Bulgarian/preprocess.yaml"), Loader=yaml.FullLoader)
sr   = pc["preprocessing"]["audio"]["sampling_rate"]
MAXV = pc["preprocessing"]["audio"]["max_wav_value"]
h = hifigan.AttrDict(json.load(open("hifigan/config.json")))

def build_gen(state_dict):
    g = hifigan.Generator(h).to(device)
    g.load_state_dict(state_dict); g.eval(); g.remove_weight_norm()
    return g

with zipfile.ZipFile("hifigan/generator_universal.pth.tar.zip") as z:
    inner = [n for n in z.namelist() if n.endswith(".pth.tar")][0]
    universal = build_gen(torch.load(io.BytesIO(z.read(inner)), map_location=device)["generator"])

best = sorted(glob.glob(f"{CP}/g_*"), key=lambda p: int(p.split('_')[-1]))[-1]
finetuned = build_gen(torch.load(best, map_location=device)["generator"])
print("finetuned checkpoint:", os.path.basename(best))

def vocode(gen, mel):
    with torch.no_grad():
        return (gen(mel).squeeze(1).cpu().numpy() * MAXV).astype("int16")[0]

for f in sorted(glob.glob("gta_mels/Bulgarian/*.npy"))[:8]:
    utt = os.path.splitext(os.path.basename(f))[0]
    mel = torch.from_numpy(np.load(f)).float().unsqueeze(0).to(device)
    print(f"================ {utt} ================")
    print("OLD universal:"); display(Audio(vocode(universal, mel), rate=sr))
    print("FINETUNED:");     display(Audio(vocode(finetuned, mel), rate=sr))
    rw = glob.glob(f"/content/data/raw/**/{utt}.wav", recursive=True)
    if rw:
        _sr, d = wavfile.read(rw[0]); print("ORIGINAL recording:"); display(Audio(d, rate=_sr))

In [ ]:

%load_ext tensorboard
%tensorboard --logdir "{CP}/logs"

In [4]:
import os, sys, glob, io, json, zipfile, yaml, torch, numpy as np
from IPython.display import Audio, display
from scipy.io import wavfile

os.chdir("/content/FastSpeech2")
sys.path.insert(0, "/content/FastSpeech2")
import hifigan   # now resolvable

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pc = yaml.load(open("config/Bulgarian/preprocess.yaml"), Loader=yaml.FullLoader)
sr   = pc["preprocessing"]["audio"]["sampling_rate"]
MAXV = pc["preprocessing"]["audio"]["max_wav_value"]
h = hifigan.AttrDict(json.load(open("hifigan/config.json")))

def build_gen(state_dict):
    g = hifigan.Generator(h).to(device)
    g.load_state_dict(state_dict); g.eval(); g.remove_weight_norm()
    return g

with zipfile.ZipFile("hifigan/generator_universal.pth.tar.zip") as z:
    inner = [n for n in z.namelist() if n.endswith(".pth.tar")][0]
    universal = build_gen(torch.load(io.BytesIO(z.read(inner)), map_location=device)["generator"])

best = sorted(glob.glob(f"{CP}/g_*"), key=lambda p: int(p.split('_')[-1]))[-1]
finetuned = build_gen(torch.load(best, map_location=device)["generator"])
print("finetuned checkpoint:", os.path.basename(best))

def vocode(gen, mel):
    with torch.no_grad():
        return (gen(mel).squeeze(1).cpu().numpy() * MAXV).astype("int16")[0]

output_dir = "/content/drive/MyDrive/fs2_bg_phone/presentation_samples"
os.makedirs(output_dir, exist_ok=True)

for f in sorted(glob.glob("gta_mels/Bulgarian/*.npy"))[:8]:
    utt = os.path.splitext(os.path.basename(f))[0]
    mel = torch.from_numpy(np.load(f)).float().unsqueeze(0).to(device)
    print(f"================ {utt} ================")
    universal_audio = vocode(universal, mel)
    finetuned_audio = vocode(finetuned, mel)
    print("OLD universal:"); display(Audio(universal_audio, rate=sr))
    print("FINETUNED:");     display(Audio(finetuned_audio, rate=sr))
    wavfile.write(os.path.join(output_dir, f"{utt}_universal.wav"), sr, universal_audio)
    wavfile.write(os.path.join(output_dir, f"{utt}_finetuned.wav"), sr, finetuned_audio)
    rw = glob.glob(f"/content/data/raw/**/{utt}.wav", recursive=True)
    if rw:
        _sr, d = wavfile.read(rw[0]); print("ORIGINAL recording:"); display(Audio(d, rate=_sr))
        wavfile.write(os.path.join(output_dir, f"{utt}_original.wav"), _sr, d)

Output hidden; open in https://colab.research.google.com to view.